In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ==========================================
# 1. PREDICTED & MEASURED ACCELERATION ENGINE
# ==========================================
MANEUVER_PROFILES = {
    "Smooth Glide & Bank Turn": {
        "peak_g_pred": 1.35,
        "turn_center": 0.48,
        "gust_amp": 0.05,
        "desc": "Steady glide with a gentle coordinated bank turn around mid-flight and minimal wind disturbance."
    },
    "Climb & Pull-Up": {
        "peak_g_pred": 2.25,
        "turn_center": 0.58,
        "gust_amp": 0.08,
        "desc": "Launch climb followed by a pull-up maneuver producing a clear positive vertical load factor peak."
    },
    "Gusty Approach": {
        "peak_g_pred": 1.65,
        "turn_center": 0.42,
        "gust_amp": 0.22,
        "desc": "Approach glide encountering atmospheric gusts and higher-frequency dynamic acceleration oscillations."
    },
}

def generate_predicted_and_simulated_imu(maneuver_name, fs_hz, duration_s, noise_level, bias_g, seed=101):
    prof = MANEUVER_PROFILES[maneuver_name]
    n_samples = max(int(fs_hz * duration_s), 60)
    t = np.linspace(0.0, duration_s, n_samples)
    u = t / max(duration_s, 1e-6)

    # Predicted vertical load factor [g] around 1.0 g equilibrium
    g_pulse = (prof["peak_g_pred"] - 1.0) * np.exp(-((u - prof["turn_center"]) / 0.11)**2)
    g_wave = prof["gust_amp"] * np.sin(2.0 * np.pi * 1.4 * u) * np.sin(np.pi * u)
    a_pred_g = 1.0 + g_pulse + g_wave

    # Simulated phone sensor reading (unmodeled gusts, slight timing/peak shift, bias, and sensor noise)
    rng = np.random.default_rng(seed)
    unmodeled_gust = (0.14 + 0.6 * prof["gust_amp"]) * np.sin(6.5 * t) * np.sin(np.pi * u)
    shifted_pulse = (prof["peak_g_pred"] - 1.0) * 1.06 * np.exp(-((u - (prof["turn_center"] + 0.025)) / 0.12)**2)
    imu_noise = rng.normal(0.0, 0.055 * max(noise_level, 0.05), size=n_samples)

    a_meas_g = 1.0 + shifted_pulse + g_wave + unmodeled_gust + bias_g + imu_noise

    return pd.DataFrame({
        "time_s": np.round(t, 3),
        "a_pred_g": np.round(a_pred_g, 4),
        "a_meas_g": np.round(a_meas_g, 4),
    })

def load_custom_or_simulated_df(source_mode, csv_path, maneuver_name, fs_hz, duration_s, noise_level, bias_g, run_idx):
    sim_df = generate_predicted_and_simulated_imu(
        maneuver_name, fs_hz, duration_s, noise_level, bias_g, seed=101 + run_idx
    )
    if source_mode == "Upload CSV in Colab":
        clean_path = csv_path.strip()
        if clean_path and os.path.exists(clean_path):
            raw = pd.read_csv(clean_path)
            t_col = next((c for c in ["time_s", "time", "t", "seconds"] if c in raw.columns), None)
            a_col = next((c for c in ["accel_g", "a_meas_g", "accel_vert_g", "az_g", "acc_g"] if c in raw.columns), None)
            if t_col and a_col:
                t_raw = raw[t_col].to_numpy(dtype=float)
                a_raw = raw[a_col].to_numpy(dtype=float)
                t_norm = np.linspace(0.0, max(t_raw[-1] - t_raw[0], 1e-3), len(sim_df))
                a_interp = np.interp(t_norm, t_raw - t_raw[0], a_raw)
                sim_df["time_s"] = np.round(t_norm, 3)
                sim_df["a_meas_g"] = np.round(a_interp, 4)
                return sim_df, f"Loaded measured acceleration from '{clean_path}' ({a_col})."
            else:
                return sim_df, f"CSV found ('{clean_path}'), but expected time/accel columns not matched; using Simulated Phone IMU."
        else:
            return sim_df, f"File '{clean_path}' not found yet; using Simulated Phone IMU fallback."
    return sim_df, "Using Simulated Phone IMU (no physical phone or outdoor flight required)."

# ==========================================
# 2. COMPARISON PLOTS & METRICS
# ==========================================
def draw_acceleration_comparison(maneuver_name, source_mode, csv_path, fs_hz, duration_s, noise_level, bias_g, run_idx):
    df, status_msg = load_custom_or_simulated_df(
        source_mode, csv_path, maneuver_name, fs_hz, duration_s, noise_level, bias_g, run_idx
    )
    t = df["time_s"].to_numpy()
    pred = df["a_pred_g"].to_numpy()
    meas = df["a_meas_g"].to_numpy()
    err = meas - pred

    rmse = float(np.sqrt(np.mean(err**2)))
    mae = float(np.mean(np.abs(err)))
    peak_diff = float(np.max(meas) - np.max(pred))

    plt.style.use("seaborn-v0_8-whitegrid")
    fig, (ax_cmp, ax_err) = plt.subplots(1, 2, figsize=(13.6, 4.6), gridspec_kw={"width_ratios": [1.3, 1.1]})

    # --- LEFT: Predicted vs. Measured Acceleration ---
    ax_cmp.plot(t, pred, color="#2563eb", lw=2.3, label="Predicted Acceleration [g]")
    ax_cmp.plot(t, meas, color="#ea580c", lw=1.5, alpha=0.88, label="Measured Acceleration [g]")
    ax_cmp.axhline(1.0, color="#64748b", ls="--", lw=1.0, label="1.0 g Baseline")

    ax_cmp.set_title(f"Predicted vs. Measured Acceleration — {maneuver_name}", fontsize=11, fontweight="bold", pad=10)
    ax_cmp.set_xlabel("Flight Time [s]", fontsize=9.5)
    ax_cmp.set_ylabel("Vertical Acceleration [g]", fontsize=9.5)
    ax_cmp.legend(loc="upper right", fontsize=8.3, frameon=True, facecolor="white")

    # --- RIGHT: Residual Error (Measured - Predicted) ---
    ax_err.plot(t, err, color="#7c3aed", lw=1.6, label="Error (Measured - Predicted)")
    ax_err.axhline(0.0, color="#0f172a", lw=1.0)
    ax_err.fill_between(t, err, 0.0, color="#c4b5fd", alpha=0.35)

    ax_err.set_title("Acceleration Residual Error [g]", fontsize=11, fontweight="bold", pad=10)
    ax_err.set_xlabel("Flight Time [s]", fontsize=9.5)
    ax_err.set_ylabel("Residual [g]", fontsize=9.5)
    ax_err.legend(loc="upper right", fontsize=8.3, frameon=True, facecolor="white")

    plt.tight_layout()
    plt.show()

    display(HTML(f"""
    <div style="
        width:100%;box-sizing:border-box;
        background:linear-gradient(135deg,#0f172a 0%,#1e293b 100%);
        border:1px solid #334155;border-left:4px solid #38bdf8;
        border-radius:8px;padding:13px 17px;color:#f8fafc;
        font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;margin-top:8px;
    ">
        <div style="font-size:13px;">
            <b>RMSE:</b> <span style="color:#7dd3fc;font-weight:700;">{rmse:.3f} g</span> &nbsp;|&nbsp;
            <b>MAE:</b> {mae:.3f} g &nbsp;|&nbsp;
            <b>Peak Diff (Max Meas - Max Pred):</b> {peak_diff:+.3f} g &nbsp;|&nbsp;
            <b>Max Measured:</b> {np.max(meas):.2f} g
        </div>
        <div style="font-size:12.5px;color:#cbd5e1;margin-top:4px;line-height:1.5;">
            {status_msg} {MANEUVER_PROFILES[maneuver_name]['desc']}
        </div>
    </div>
    """))

# ==========================================
# 3. INTERACTIVE ENGLISH UI
# ==========================================
lbl_style = {"description_width": "145px"}
ctrl_layout = widgets.Layout(width="340px")

w_man  = widgets.Dropdown(options=list(MANEUVER_PROFILES.keys()), value="Climb & Pull-Up", description="Maneuver:", style=lbl_style, layout=ctrl_layout)
w_src  = widgets.Dropdown(options=["Simulated Phone IMU", "Upload CSV in Colab"], value="Simulated Phone IMU", description="Measured Source:", style=lbl_style, layout=ctrl_layout)
w_csv  = widgets.Text(value="phone_flight_accel.csv", description="CSV File Path:", style=lbl_style, layout=ctrl_layout)
w_fs   = widgets.FloatSlider(value=50.0, min=10.0, max=200.0, step=10.0, description="Sample Rate [Hz]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_dur  = widgets.FloatSlider(value=20.0, min=5.0,  max=60.0,  step=5.0,  description="Duration [s]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_nois = widgets.FloatSlider(value=1.0,  min=0.0,  max=3.0,   step=0.2,  description="IMU Noise:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)
w_bias = widgets.FloatSlider(value=0.05, min=-0.3, max=0.3,   step=0.02, description="Sensor Bias [g]:", readout_format="+.2f", style=lbl_style, layout=ctrl_layout)

btn_cmp = widgets.Button(description="Compare Acceleration", button_style="primary", icon="line-chart", layout=widgets.Layout(width="220px", height="36px"))
out_cmp = widgets.Output(layout=widgets.Layout(width="100%"))
run_cnt = {"n": 0}

def refresh_cmp(*args):
    run_cnt["n"] += 1
    with out_cmp:
        clear_output(wait=True)
        draw_acceleration_comparison(
            w_man.value, w_src.value, w_csv.value, w_fs.value,
            w_dur.value, w_nois.value, w_bias.value, run_cnt["n"]
        )

btn_cmp.on_click(refresh_cmp)
for c in [w_man, w_src, w_csv, w_fs, w_dur, w_nois, w_bias]:
    c.observe(refresh_cmp, names="value")

header_cmp = widgets.HTML("""
    <div style="
        width:100%;box-sizing:border-box;
        background:linear-gradient(135deg,#0f172a 0%,#1e293b 100%);
        border:1px solid #334155;border-left:4px solid #38bdf8;
        border-radius:8px;padding:12px 16px;margin-bottom:10px;
        font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
    ">
        <div style="font-size:17px;font-weight:700;color:#f8fafc;">Model-Flight Acceleration — Measured vs. Predicted</div>
        <div style="font-size:12px;color:#94a3b8;margin-top:2px;">
            Compare predicted flight acceleration with simulated phone IMU data or an uploaded CSV log in Colab
        </div>
    </div>
""")

controls_cmp = widgets.VBox([
    widgets.HBox([
        widgets.VBox([w_man, w_src, w_csv, w_fs], layout=widgets.Layout(margin="0 28px 0 0", gap="8px")),
        widgets.VBox([w_dur, w_nois, w_bias, btn_cmp], layout=widgets.Layout(gap="8px"))
    ], layout=widgets.Layout(align_items="flex-start", flex_flow="row wrap"))
], layout=widgets.Layout(
    width="100%", padding="14px 16px", border="1px solid #334155",
    border_radius="8px", background_color="#1e293b", margin="0 0 12px 0"
))

display(header_cmp, controls_cmp, out_cmp)
refresh_cmp()

HTML(value='\n    <div style="\n        width:100%;box-sizing:border-box;\n        background:linear-gradient(…

Output(layout=Layout(width='100%'))

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ==========================================
# 1. PREDICTED & MEASURED ACCELERATION ENGINE
# ==========================================
MANEUVER_PROFILES = {
    "Smooth Glide & Bank Turn": {
        "peak_g_pred": 1.35,
        "turn_center": 0.48,
        "gust_amp": 0.05,
        "desc": "Steady glide with a gentle coordinated bank turn around mid-flight and minimal wind disturbance."
    },
    "Climb & Pull-Up": {
        "peak_g_pred": 2.25,
        "turn_center": 0.58,
        "gust_amp": 0.08,
        "desc": "Launch climb followed by a pull-up maneuver producing a clear positive vertical load factor peak."
    },
    "Gusty Approach": {
        "peak_g_pred": 1.65,
        "turn_center": 0.42,
        "gust_amp": 0.22,
        "desc": "Approach glide encountering atmospheric gusts and higher-frequency dynamic acceleration oscillations."
    },
}

def build_comparison_df(maneuver_name, meas_mode, manual_peak_g, fs_hz, duration_s, noise_level, bias_g, seed=101):
    prof = MANEUVER_PROFILES[maneuver_name]
    n_samples = max(int(fs_hz * duration_s), 60)
    t = np.linspace(0.0, duration_s, n_samples)
    u = t / max(duration_s, 1e-6)

    g_pulse = (prof["peak_g_pred"] - 1.0) * np.exp(-((u - prof["turn_center"]) / 0.11)**2)
    g_wave = prof["gust_amp"] * np.sin(2.0 * np.pi * 1.4 * u) * np.sin(np.pi * u)
    a_pred_g = 1.0 + g_pulse + g_wave

    rng = np.random.default_rng(seed)
    imu_noise = rng.normal(0.0, 0.055 * max(noise_level, 0.0), size=n_samples)

    if meas_mode == "Manual Peak Value [g]":
        pulse_shape = np.exp(-((u - prof["turn_center"]) / 0.12)**2)
        pulse_shape = pulse_shape / max(np.max(pulse_shape), 1e-9)
        target_amp = manual_peak_g - 1.0
        a_meas_g = 1.0 + target_amp * pulse_shape + bias_g + imu_noise
        source_note = f"Manual measured peak input ({manual_peak_g:.2f} g)"
    else:
        unmodeled_gust = (0.14 + 0.6 * prof["gust_amp"]) * np.sin(6.5 * t) * np.sin(np.pi * u)
        shifted_pulse = (prof["peak_g_pred"] - 1.0) * 1.06 * np.exp(-((u - (prof["turn_center"] + 0.025)) / 0.12)**2)
        a_meas_g = 1.0 + shifted_pulse + g_wave + unmodeled_gust + bias_g + imu_noise
        source_note = "Simulated phone IMU measurement"

    return pd.DataFrame({
        "time_s": np.round(t, 3),
        "a_pred_g": np.round(a_pred_g, 4),
        "a_meas_g": np.round(a_meas_g, 4),
    }), source_note

def build_verbal_explanation(pred, meas, err, rmse, peak_diff, noise_level, bias_g, source_note):
    reasons = []

    if abs(peak_diff) < 0.08:
        reasons.append("the peak acceleration closely matches the predicted maneuver load")
    elif peak_diff > 0:
        reasons.append(f"the measured peak ({np.max(meas):.2f} g) is higher than predicted ({np.max(pred):.2f} g), indicating stronger pull/turn load or upward gust/noise contribution")
    else:
        reasons.append(f"the measured peak ({np.max(meas):.2f} g) is lower than predicted ({np.max(pred):.2f} g), indicating a gentler maneuver or smaller load buildup")

    if abs(bias_g) >= 0.04:
        direction = "upward" if bias_g > 0 else "downward"
        reasons.append(f"a non-zero sensor bias ({bias_g:+.2f} g) shifts the baseline {direction}")

    if noise_level >= 1.2:
        reasons.append(f"elevated IMU noise ({noise_level:.1f}) adds high-frequency scatter and raises RMSE ({rmse:.3f} g)")
    elif rmse < 0.12:
        reasons.append(f"overall agreement is strong with a low RMSE of {rmse:.3f} g")
    else:
        reasons.append(f"dynamic fluctuations/timing differences produce a moderate RMSE of {rmse:.3f} g")

    return f"<b>Why This Result Happened ({source_note}):</b> The curve looks like this because " + "; and ".join(reasons) + "."

# ==========================================
# 2. COMPARISON PLOTS, METRICS & EXPLANATION
# ==========================================
def draw_acceleration_comparison(maneuver_name, meas_mode, manual_peak_g, fs_hz, duration_s, noise_level, bias_g, run_idx):
    df, source_note = build_comparison_df(
        maneuver_name, meas_mode, manual_peak_g, fs_hz, duration_s, noise_level, bias_g, seed=101 + run_idx
    )
    t = df["time_s"].to_numpy()
    pred = df["a_pred_g"].to_numpy()
    meas = df["a_meas_g"].to_numpy()
    err = meas - pred

    rmse = float(np.sqrt(np.mean(err**2)))
    mae = float(np.mean(np.abs(err)))
    peak_diff = float(np.max(meas) - np.max(pred))
    why_text = build_verbal_explanation(pred, meas, err, rmse, peak_diff, noise_level, bias_g, source_note)

    plt.style.use("seaborn-v0_8-whitegrid")
    fig, (ax_cmp, ax_err) = plt.subplots(1, 2, figsize=(13.6, 4.6), gridspec_kw={"width_ratios": [1.3, 1.1]})

    ax_cmp.plot(t, pred, color="#2563eb", lw=2.3, label="Predicted Acceleration [g]")
    ax_cmp.plot(t, meas, color="#ea580c", lw=1.6, alpha=0.90, label="Measured Acceleration [g]")
    ax_cmp.axhline(1.0, color="#64748b", ls="--", lw=1.0, label="1.0 g Baseline")

    ax_cmp.set_title(f"Predicted vs. Measured Acceleration — {maneuver_name}", fontsize=11, fontweight="bold", pad=10)
    ax_cmp.set_xlabel("Flight Time [s]", fontsize=9.5)
    ax_cmp.set_ylabel("Vertical Acceleration [g]", fontsize=9.5)
    ax_cmp.legend(loc="upper right", fontsize=8.3, frameon=True, facecolor="white")

    ax_err.plot(t, err, color="#7c3aed", lw=1.6, label="Error (Measured - Predicted)")
    ax_err.axhline(0.0, color="#0f172a", lw=1.0)
    ax_err.fill_between(t, err, 0.0, color="#c4b5fd", alpha=0.35)

    ax_err.set_title("Acceleration Residual Error [g]", fontsize=11, fontweight="bold", pad=10)
    ax_err.set_xlabel("Flight Time [s]", fontsize=9.5)
    ax_err.set_ylabel("Residual [g]", fontsize=9.5)
    ax_err.legend(loc="upper right", fontsize=8.3, frameon=True, facecolor="white")

    plt.tight_layout()
    plt.show()

    display(HTML(f"""
    <div style="
        width:100%;box-sizing:border-box;
        background:linear-gradient(135deg,#0f172a 0%,#1e293b 100%);
        border:1px solid #334155;border-left:4px solid #38bdf8;
        border-radius:8px;padding:13px 17px;color:#f8fafc;
        font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;margin-top:8px;
    ">
        <div style="font-size:13px;">
            <b>RMSE:</b> <span style="color:#7dd3fc;font-weight:700;">{rmse:.3f} g</span> &nbsp;|&nbsp;
            <b>MAE:</b> {mae:.3f} g &nbsp;|&nbsp;
            <b>Peak Diff (Max Meas - Max Pred):</b> {peak_diff:+.3f} g &nbsp;|&nbsp;
            <b>Max Measured:</b> {np.max(meas):.2f} g
        </div>
        <div style="font-size:12.5px;color:#e2e8f0;margin-top:6px;line-height:1.55;">
            {why_text}
        </div>
        <div style="font-size:12px;color:#94a3b8;margin-top:4px;line-height:1.45;">
            {MANEUVER_PROFILES[maneuver_name]['desc']}
        </div>
    </div>
    """))

# ==========================================
# 3. INTERACTIVE ENGLISH UI (NO CSV INPUT)
# ==========================================
lbl_style = {"description_width": "155px"}
ctrl_layout = widgets.Layout(width="345px")

w_man   = widgets.Dropdown(options=list(MANEUVER_PROFILES.keys()), value="Climb & Pull-Up", description="Maneuver:", style=lbl_style, layout=ctrl_layout)
w_mode  = widgets.Dropdown(options=["Simulated Phone IMU", "Manual Peak Value [g]"], value="Manual Peak Value [g]", description="Measured Mode:", style=lbl_style, layout=ctrl_layout)
w_man_g = widgets.FloatSlider(value=1.80, min=0.20, max=4.50, step=0.05, description="Manual Peak [g]:", readout_format=".2f", style=lbl_style, layout=ctrl_layout)
w_fs    = widgets.FloatSlider(value=50.0, min=10.0, max=200.0, step=10.0, description="Sample Rate [Hz]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_dur   = widgets.FloatSlider(value=20.0, min=5.0,  max=60.0,  step=5.0,  description="Duration [s]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_nois  = widgets.FloatSlider(value=0.6,  min=0.0,  max=3.0,   step=0.2,  description="IMU Noise:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)
w_bias  = widgets.FloatSlider(value=0.00, min=-0.4, max=0.4,   step=0.02, description="Sensor Bias [g]:", readout_format="+.2f", style=lbl_style, layout=ctrl_layout)

btn_cmp = widgets.Button(description="Compare Acceleration", button_style="primary", icon="line-chart", layout=widgets.Layout(width="220px", height="36px"))
out_cmp = widgets.Output(layout=widgets.Layout(width="100%"))
run_cnt = {"n": 0}

def refresh_cmp(*args):
    run_cnt["n"] += 1
    with out_cmp:
        clear_output(wait=True)
        draw_acceleration_comparison(
            w_man.value, w_mode.value, w_man_g.value, w_fs.value,
            w_dur.value, w_nois.value, w_bias.value, run_cnt["n"]
        )

btn_cmp.on_click(refresh_cmp)
for c in [w_man, w_mode, w_man_g, w_fs, w_dur, w_nois, w_bias]:
    c.observe(refresh_cmp, names="value")

header_cmp = widgets.HTML("""
    <div style="
        width:100%;box-sizing:border-box;
        background:linear-gradient(135deg,#0f172a 0%,#1e293b 100%);
        border:1px solid #334155;border-left:4px solid #38bdf8;
        border-radius:8px;padding:12px 16px;margin-bottom:10px;
        font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
    ">
        <div style="font-size:17px;font-weight:700;color:#f8fafc;">Model-Flight Acceleration — Measured vs. Predicted</div>
        <div style="font-size:12px;color:#94a3b8;margin-top:2px;">
            Compare predicted flight acceleration against manual measured peak input or simulated IMU data
        </div>
    </div>
""")

controls_cmp = widgets.VBox([
    widgets.HBox([
        widgets.VBox([w_man, w_mode, w_man_g, w_fs], layout=widgets.Layout(margin="0 28px 0 0", gap="8px")),
        widgets.VBox([w_dur, w_nois, w_bias, btn_cmp], layout=widgets.Layout(gap="8px"))
    ], layout=widgets.Layout(align_items="flex-start", flex_flow="row wrap"))
], layout=widgets.Layout(
    width="100%", padding="14px 16px", border="1px solid #334155",
    border_radius="8px", background_color="#1e293b", margin="0 0 12px 0"
))

display(header_cmp, controls_cmp, out_cmp)
refresh_cmp()

HTML(value='\n    <div style="\n        width:100%;box-sizing:border-box;\n        background:linear-gradient(…

Output(layout=Layout(width='100%'))